In [1]:
from google.colab import files

uploaded = files.upload()

Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset.csv


In [2]:
import numpy as np
import pandas as pd

In [3]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Original Dataset Shape:", df.shape)
display(df.head())

Original Dataset Shape: (50000, 32)


,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly,Salary Package
0,1,Male,Ahmedabad,Tier2,ECE,Networking,No,No,6.02,6.54,...,0,66.7,2.2,49.4,47.8,0,Low,0,0,0.00
1,2,Female,Mumbai,Tier2,ECE,DataScience,Yes,Yes,5.84,5.12,...,0,48.2,2.4,26.7,25.8,0,Low,0,0,0.00
2,3,Male,Kolkata,Tier2,IT,DataScience,Yes,No,4.91,5.29,...,0,73.8,2.8,67.7,41.5,0,Low,1,0,3.89
3,4,Male,Jaipur,Tier1,CS,AI,No,No,7.67,8.03,...,0,69.8,2.7,66.9,48.0,0,Mid,1,0,8.37
4,5,Male,Pune,Tier2,IT,DataScience,Yes,No,8.14,8.97,...,1,73.1,2.1,71.7,61.7,1,High,1,0,18.99


In [4]:
missing = df.isnull().sum().sort_values(ascending=False)

print("Missing Values:")
display(missing[missing > 0])

print("Total Missing Values:", df.isnull().sum().sum())

Missing Values:


,0
MockInterviewScore,4957
Workshops,4488
AptitudeTestScore,4029
SoftSkillsRating,3526
CodingTestScore,2976


Total Missing Values: 19976


In [5]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [6]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape After Removing Duplicates:", df.shape)

Shape After Removing Duplicates: (50000, 32)


In [7]:
target = "PlacementStatus"

X = df.drop(columns=[target, "StudentID", "Salary Package"])
y = df[target]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (50000, 29)
Target Shape: (50000,)


In [8]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nNumber of Numerical Features:", len(numeric_features))
print("Number of Categorical Features:", len(categorical_features))

Numerical Features:
['SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'IsAnomaly']

Categorical Features:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA_Tier']

Number of Numerical Features: 21
Number of Categorical Features: 8


In [9]:
X = X.replace([np.inf, -np.inf], np.nan)

print("Total Missing Values After Infinity Check:", X.isnull().sum().sum())

Total Missing Values After Infinity Check: 19976


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [11]:
X_clean = preprocessor.fit_transform(X)

print("Cleaned Feature Shape:", X_clean.shape)
print("Missing Values:", np.isnan(X_clean).sum())

Cleaned Feature Shape: (50000, 53)
Missing Values: 0


In [12]:
feature_names = preprocessor.get_feature_names_out()

print("Number of Processed Features:", len(feature_names))
print("\nFirst 20 Processed Features:")
print(feature_names[:20])

Number of Processed Features: 53

First 20 Processed Features:
['numeric__SGPA_Sem1' 'numeric__SGPA_Sem2' 'numeric__SGPA_Sem3'
 'numeric__SGPA_Sem4' 'numeric__SGPA_Sem5' 'numeric__SGPA_Sem6'
 'numeric__SGPA_Sem7' 'numeric__SGPA_Sem8' 'numeric__CGPA'
 'numeric__AttendancePercent' 'numeric__Internships' 'numeric__Projects'
 'numeric__Workshops' 'numeric__Certifications' 'numeric__Publications'
 'numeric__AptitudeTestScore' 'numeric__SoftSkillsRating'
 'numeric__CodingTestScore' 'numeric__MockInterviewScore'
 'numeric__ExtraCurricular']


In [13]:
cleaned_df = pd.DataFrame(
    X_clean,
    columns=feature_names
)

cleaned_df["PlacementStatus"] = y.values

print("Final Cleaned Dataset Shape:", cleaned_df.shape)
display(cleaned_df.head())

Final Cleaned Dataset Shape: (50000, 54)


,numeric__SGPA_Sem1,numeric__SGPA_Sem2,numeric__SGPA_Sem3,numeric__SGPA_Sem4,numeric__SGPA_Sem5,numeric__SGPA_Sem6,numeric__SGPA_Sem7,numeric__SGPA_Sem8,numeric__CGPA,numeric__AttendancePercent,...,categorical__Specialisation_Embedded,categorical__Specialisation_Networking,categorical__Hostel_No,categorical__Hostel_Yes,categorical__HistoryOfBacklogs_No,categorical__HistoryOfBacklogs_Yes,categorical__CGPA_Tier_High,categorical__CGPA_Tier_Low,categorical__CGPA_Tier_Mid,PlacementStatus
0,-0.784397,-0.445330,-0.461066,-0.794142,-0.496683,-0.851093,-0.488854,-1.094993,-0.591800,-0.273352,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0
1,-0.902988,-1.372228,-1.221563,-1.316232,-1.403591,-1.462354,-1.161034,-1.320865,-1.247875,-1.780455,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0
2,-1.515704,-1.261261,-1.124890,-0.966050,-1.214651,-0.869805,-0.902029,-0.991214,-1.068383,0.071358,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1
3,0.302680,0.527259,0.544337,0.250036,-0.251062,-0.108849,0.170993,0.058786,0.163305,-0.634095,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1
4,0.612332,1.140839,0.724794,0.829428,0.813296,0.689532,1.120678,0.956169,0.868894,1.530362,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1


In [14]:
print("Total Missing Values:", cleaned_df.isnull().sum().sum())
print("Total Duplicate Rows:", cleaned_df.duplicated().sum())

Total Missing Values: 0
Total Duplicate Rows: 0


In [15]:
print("Placement Status Distribution:")
print(cleaned_df["PlacementStatus"].value_counts())

print("\nPlacement Status Percentage:")
print(cleaned_df["PlacementStatus"].value_counts(normalize=True) * 100)

Placement Status Distribution:
PlacementStatus
1    32856
0    17144
Name: count, dtype: int64

Placement Status Percentage:
PlacementStatus
1    65.712
0    34.288
Name: proportion, dtype: float64


In [16]:
print("Final Dataset Information:")
cleaned_df.info()

Final Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 54 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   numeric__SGPA_Sem1                       50000 non-null  float64
 1   numeric__SGPA_Sem2                       50000 non-null  float64
 2   numeric__SGPA_Sem3                       50000 non-null  float64
 3   numeric__SGPA_Sem4                       50000 non-null  float64
 4   numeric__SGPA_Sem5                       50000 non-null  float64
 5   numeric__SGPA_Sem6                       50000 non-null  float64
 6   numeric__SGPA_Sem7                       50000 non-null  float64
 7   numeric__SGPA_Sem8                       50000 non-null  float64
 8   numeric__CGPA                            50000 non-null  float64
 9   numeric__AttendancePercent               50000 non-null  float64
 10  numeric__Internship

In [17]:
output_file = "placement_predict_50k_cleaned.csv"

cleaned_df.to_csv(output_file, index=False)

print("Cleaned dataset saved as:", output_file)

Cleaned dataset saved as: placement_predict_50k_cleaned.csv


In [18]:
files.download("placement_predict_50k_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
print("""
SESSION 2 — PREPROCESSING PIPELINE SUMMARY

1. Loaded the raw placement dataset.
2. Checked missing values.
3. Removed duplicate records.
4. Separated features and target.
5. Removed identifier and non-input columns.
6. Identified numerical and categorical features.
7. Replaced infinite values with missing values.
8. Applied median imputation to numerical features.
9. Applied most-frequent imputation to categorical features.
10. Applied standard scaling to numerical features.
11. Applied one-hot encoding to categorical features.
12. Verified that the cleaned data contains no missing values.
13. Exported the cleaned ML-ready dataset.
""")


SESSION 2 — PREPROCESSING PIPELINE SUMMARY

1. Loaded the raw placement dataset.
2. Checked missing values.
3. Removed duplicate records.
4. Separated features and target.
5. Removed identifier and non-input columns.
6. Identified numerical and categorical features.
7. Replaced infinite values with missing values.
8. Applied median imputation to numerical features.
9. Applied most-frequent imputation to categorical features.
10. Applied standard scaling to numerical features.
11. Applied one-hot encoding to categorical features.
12. Verified that the cleaned data contains no missing values.
13. Exported the cleaned ML-ready dataset.

